# Horizon-shift signal vs foreground filtering

Antenna-position error changes the antenna temperature by $\Delta T_\mathrm{ant}(\nu)$. Here we ask how *foreground-like* that change is: we project the $\Delta T_\mathrm{ant}$ spectra (top row, +1 m East/North/Up at 24 LSTs) onto the foreground spectral modes -- the same SVD modes as `foreground_svd.npz` -- and plot the residual RMS after filtering the leading $N$ modes (bottom row). The signal sits in the same low-order foreground subspace: every LST and axis falls below 10 mK within a handful of modes, so the same foreground filtering that cleans the sky also removes the position-error systematic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

In [ ]:
d = np.load("horizon_shift.npz", allow_pickle=True)
freqs = d["freqs_MHz"]
lst = d["lst_hr"]                 # LST [h] of each plotted spectrum
dT = d["dT_spectra"]             # (3, n_lst, n_freq) uncorrected dT_ant [K]
Vh = d["Vh"]                     # (n_freq, n_freq) foreground spectral modes
labels = [str(s) for s in d["labels"]]
n_f = freqs.size
N_SHOW = 18                       # foreground modes filtered (x-axis)
print(dT.shape, "spectra at LSTs", np.round(lst, 1))

In [ ]:
CMAP, norm = "twilight", Normalize(0, 24)
cmap = plt.get_cmap(CMAP)
n_modes = np.arange(N_SHOW + 1)


def resid_curves(dT_axis):
    """Per-LST residual RMS over freq [K] after filtering the leading N modes."""
    coeff = dT_axis @ Vh.T                                  # (n_lst, n_freq)
    return np.array([np.sqrt(np.sum(coeff[:, N:] ** 2, axis=1) / n_f)
                     for N in n_modes])                     # (N_SHOW+1, n_lst)


fig, axes = plt.subplots(
    2, 3, figsize=(7.3, 3.5),
    gridspec_kw=dict(height_ratios=[1.7, 1]),
    layout="constrained",
)
for col, lab in enumerate(labels):
    at, ab = axes[0, col], axes[1, col]
    for j in range(lst.size):                               # top: dT(nu) spectra
        at.plot(freqs, dT[col, j], color=cmap(norm(lst[j])), lw=0.7, alpha=0.9)
    at.axhline(0, color="0.5", lw=0.6, ls="--", zorder=0)
    at.set_title(lab, fontsize=8.5)
    at.set_xlabel("Frequency [MHz]", fontsize=8)
    at.grid(alpha=0.2); at.tick_params(labelsize=7)

    rc = resid_curves(dT[col])                              # bottom: filtered residual
    for j in range(lst.size):
        ab.plot(n_modes, rc[:, j], color=cmap(norm(lst[j])), lw=0.7, alpha=0.9)
    ab.axhline(1e-2, color="red", lw=1.2, ls="--", alpha=0.8)
    ab.set_yscale("log")
    ab.set_xlabel("Foreground modes filtered", fontsize=8)
    ab.grid(True, which="both", ls=":", lw=0.5, alpha=0.6)
    ab.set_xlim(0, N_SHOW); ab.set_ylim(1e-4, 5); ab.tick_params(labelsize=7)

axes[0, 0].set_ylabel(r"$\Delta T_\mathrm{ant}$ [K]", fontsize=8)
axes[1, 0].set_ylabel("Residual RMS [K]", fontsize=8)
axes[1, 2].text(N_SHOW, 1.2e-2, "10 mK", color="red", alpha=0.8,
                fontsize=7, va="bottom", ha="right")
for col in (1, 2):
    axes[1, col].tick_params(labelleft=False)

sm = ScalarMappable(norm=norm, cmap=CMAP)
cb = fig.colorbar(sm, ax=axes, pad=0.012, fraction=0.022)
cb.set_label("LST [h]", fontsize=8); cb.set_ticks(np.arange(0, 25, 4))
cb.ax.tick_params(labelsize=7)
fig.savefig("horizon_shift.pdf", bbox_inches="tight", dpi=600)

In [ ]:
for col, lab in enumerate(labels):
    rc = resid_curves(dT[col])               # (N_SHOW+1, n_lst)
    worst = rc.max(axis=1)                    # worst LST at each N
    n10 = int(np.nonzero(worst < 1e-2)[0][0])
    print(f"{lab:11s}  full RMS {worst[0]*1e3:7.1f} mK  ->  "
          f"all LSTs < 10 mK after {n10} modes; "
          f"residual after 10 modes {worst[10]*1e3:.2f} mK")